# 🔬 Notebook 3: Reddit — Deep Dives


## 🎯 Learning objectives

- Understand the **Hot ranking** formula piece by piece and see why it beats naïve alternatives.
- See a real lock-contention bottleneck on a single counter and fix it with **sharded counters**.
- Serve a comment tree efficiently: compare recursive parent_id queries to **materialized path**
  and **closure tables**.
- Every deep-dive follows the repo convention: **bad → better → best** with runnable code.


## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Hot ranking — what goes to the top?

A Hot feed must balance **quality** (lots of upvotes) and **freshness** (posted recently).
If we pick either signal alone we lose.


### 🐌 Bad — sort by net upvotes


In [ ]:
import random, math, time

random.seed(7)

# 10 posts: (id, ups, downs, age_hours)
posts = [
    ("A-brand-new",       5,    0,  0.0),   # new, few votes
    ("B-one-hour",       80,    5,  1.0),
    ("C-six-hours",     500,   40,  6.0),
    ("D-one-day",      3000,  200, 24.0),
    ("E-three-days",  20000, 1000, 72.0),   # old but huge
    ("F-controversial", 300,  290,  2.0),
    ("G-fresh-good",    150,    2,  0.5),
    ("H-week-old",    50000,  500, 24*7.),  # a week old classic
    ("I-meh",             8,    6, 10.0),
    ("J-rising",         60,    1,  0.25),
]

print("SORT BY NET UPVOTES ONLY (ignores age):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -(p[1]-p[2])):
    print(f"  net={u-d:>6}  age={age_h:>5.1f}h  {pid}")


Problem: **H-week-old** and **E-three-days** dominate forever. A brand-new viral post
(J-rising) looks tiny next to them. Reddit would ossify.


### ✅ Better — Hacker News style exponential decay


In [ ]:
# Hacker News: score = (ups - 1) / (age_hours + 2) ** 1.8
def hn_score(ups, downs, age_h):
    return (max(ups - downs - 1, 0)) / ((age_h + 2) ** 1.8)

print("HACKER-NEWS SCORE (exponential decay):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -hn_score(p[1], p[2], p[3])):
    print(f"  score={hn_score(u,d,age_h):>8.3f}  age={age_h:>5.1f}h  {pid}")


Much better — recent posts can dethrone old classics. But decay is *continuous*: every
minute that passes recomputes every post's score, otherwise the ranking drifts. Expensive.


### 🏆 Best — Reddit's log + post-time formula


In [ ]:
# Reddit's hot formula, transcribed from the open-sourced `_sorts.pyx`:
#
#   s       = ups - downs
#   order   = log10(max(|s|, 1))
#   sign    = +1 if s > 0 else -1 if s < 0 else 0
#   seconds = epoch_seconds(created_at) - 1134028003     # Reddit's launch, 2005-12-08
#   hot     = round(sign * order + seconds / 45000, 7)
#
# Read the sign placement carefully — it is the part everyone gets wrong.
# `sign` multiplies the LOG term, NOT the time term. Write `order + sign*seconds/45000`
# by mistake and a single net-downvoted post drops to roughly MINUS 39,000 while every
# other post sits near +14,500: your entire sort collapses into "positive vs negative".
# We demonstrate that failure mode below.
#
# Three properties the real formula gives us:
#   • log-scaling: 10,000 votes is not 10x better than 1,000 — it is +1 on the score.
#   • Newer posts start from a HIGHER baseline (larger `seconds`), so fresh content rises.
#   • For fixed votes the score of an existing post NEVER changes — so a new vote
#     re-scores only THAT post. No global decay sweep. This is the trick that makes the
#     Hot feed cheap to maintain.

import math, time as _time

REDDIT_EPOCH = 1_134_028_003          # 2005-12-08 07:46:43 UTC
NOW_T = int(_time.time())

def created_at_of(age_seconds: float) -> int:
    """Unix timestamp of a post that was created `age_seconds` ago."""
    return NOW_T - int(age_seconds)

def hot_score(ups: int, downs: int, age_seconds: float) -> float:
    net = ups - downs
    sign = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    seconds = created_at_of(age_seconds) - REDDIT_EPOCH
    return round(sign * order + seconds / 45_000, 7)

print("REDDIT HOT SCORE (sign*log10(|net|) + seconds_since_epoch/45000):")
for pid, u, d, age_h in sorted(posts, key=lambda p: -hot_score(p[1], p[2], p[3] * 3600)):
    print(f"  score={hot_score(u, d, age_h*3600):>14.3f}  age={age_h:>5.1f}h  net={u-d:>6}  {pid}")


Observe the ordering:

- Fresh, well-upvoted posts (`G-fresh-good`, `J-rising`) land near the top.
- Week-old classics (`H-week-old`) fall away despite having the most votes — their smaller
  `seconds` term is a **12.5-hours-per-log10-of-votes** drag.
- Scores of *existing* posts don't change over time for fixed votes, so a new vote re-scores
  exactly one post. That's the whole trick.

### Why the sign goes on the log term

Let's put a spam post (5 up, 900 down) through both the correct formula and the common
mis-transcription, and watch the mis-transcription blow up.

In [ ]:
def hot_wrong(ups, downs, age_seconds):
    """The common mis-transcription: sign applied to the TIME term instead of the log term."""
    net = ups - downs
    sign = 1 if net > 0 else (-1 if net < 0 else 0)
    order = math.log10(max(abs(net), 1))
    seconds = created_at_of(age_seconds) - REDDIT_EPOCH
    return round(order + sign * seconds / 45_000, 7)

AGE = 0.5 * 3600                      # compare at one fixed age so only the votes differ
baseline = hot_score(1, 0, AGE)       # a same-age post with net +1

def penalty_hours(fn, ups, downs):
    """How much recency (in hours) this post gives up versus a same-age net+1 post."""
    return (baseline - fn(ups, downs, AGE)) * 45_000 / 3600

print(f"{'post':<22}{'correct':>14}{'mis-transcribed':>18}")
for label, u, d in (("spam  (net  -895)", 5, 900), ("good  (net   +59)", 60, 1)):
    print(f"  {label:<20}{hot_score(u, d, AGE):>14.1f}{hot_wrong(u, d, AGE):>18.1f}")

print(f"\nRecency given up versus a same-age net+1 post:")
for label, u, d in (("spam  (net  -895)", 5, 900), ("good  (net   +59)", 60, 1)):
    print(f"  {label:<20} correct: {penalty_hours(hot_score, u, d):>12,.1f} h"
          f"   mis-transcribed: {penalty_hours(hot_wrong, u, d):>14,.0f} h")

print("""
Correct: the spam post gives up ~3 log units ≈ 37 hours of recency. It sinks hard, but it stays
on the same number line as everything else, so a post that was wrongly buried can climb back
once the votes turn around.

Mis-transcribed: the spam post gives up the number printed above — tens of thousands of
years. Every net-negative post is
now unreachably below every net-positive post, and — worse — among negative posts the ordering
is INVERTED, because a larger `seconds` is being subtracted: older spam outranks newer spam.

The bug is invisible until one post goes net-negative, which is exactly why a ranking formula
needs a downvoted fixture in its tests.""")


In [ ]:
# The /45000 constant is the exchange rate between votes and freshness.
# 45,000 seconds = 12.5 hours. So one extra log10 of net votes (a 10x vote bump)
# should be worth exactly 12.5 hours of age. Let's prove it.
a_score = hot_score(ups=10_000, downs=0, age_seconds=12.5 * 3600)   # older, 10x the votes
b_score = hot_score(ups=1_000,  downs=0, age_seconds=0)             # brand-new, 1x the votes
print(f"A (12.5h old, 10k votes): {a_score:.4f}")
print(f"B ( 0.0h old,  1k votes): {b_score:.4f}")
print(f"difference              : {b_score - a_score:+.4f}   (≈ 0 → correctly tied)")
assert abs(b_score - a_score) < 1e-3, "10x votes must buy exactly 12.5 hours"
print("\nTune 45000 to change the site's personality: smaller = more churn, larger = more stable.")


## 1️⃣b The other sorts — Top, Controversial, and Best

"Hot" is one of five sorts, and each answers a genuinely different question. Two of them have
real published formulas that are worth knowing, because they show up verbatim in interviews.

| Sort | Question | Formula |
|---|---|---|
| **New** | what just arrived? | `ORDER BY created_at DESC` |
| **Top** | what won, all-time / this week? | `ORDER BY (ups - downs) DESC` within a time window |
| **Hot** | what is winning *right now*? | `sign*log10(|net|) + seconds/45000` |
| **Controversial** | what is people *fighting* about? | `(ups+downs) ** balance`, `balance = min(u,d)/max(u,d)` |
| **Best** (comments) | which comment is genuinely good? | Wilson score lower bound |

**Top needs a time window.** `?t=day|week|month|year|all` is not a UI nicety — without it Top
is a frozen monument and the query can be answered from a materialised view per window.

**Controversial** rewards posts that are simultaneously *high volume* and *evenly split*. The
exponent is what makes it work: a 500/500 split raises magnitude to the power 1, while a 999/1
split raises it to the power ~0.001, which collapses to ≈1 no matter how big the post is.

**Best** is the comment sort, and it is the one that needs statistics. A comment at 1↑/0↓ has a
100% upvote rate but one sample; a comment at 400↑/100↓ has 80% but is *certain*. Sorting by raw
ratio puts the 1-vote comment on top forever. The **Wilson score lower bound** asks instead:
"given what we've seen, what is the *worst* true upvote rate consistent with this data at 80%
confidence?" — which naturally penalises small samples.

In [ ]:
# All three transcribed from Reddit's open-sourced sorting code.

def top_score(ups, downs):
    """Top = raw net score. Always paired with a time window in the query."""
    return ups - downs

def controversy(ups, downs):
    """Reddit's _controversy: magnitude ** balance."""
    if downs <= 0 or ups <= 0:
        return 0.0                       # one-sided posts are not controversial
    magnitude = ups + downs
    balance = (downs / ups) if ups > downs else (ups / downs)   # in (0, 1]
    return magnitude ** balance

def confidence(ups, downs, z=1.281551565545):
    """Wilson score lower bound at 80% confidence — Reddit's 'best' comment sort."""
    n = ups + downs
    if n == 0:
        return 0.0
    p = ups / n
    left  = p + z * z / (2 * n)
    right = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    under = 1 + z * z / n
    return (left - right) / under

# --- Controversial: which of our posts is the flame war?
print("CONTROVERSIAL:")
for pid, u, d, age_h in sorted(posts, key=lambda p: -controversy(p[1], p[2]))[:4]:
    bal = (min(u, d) / max(u, d)) if u and d else 0
    print(f"  {controversy(u,d):>10.1f}  {u:>6}↑/{d:>5}↓  balance={bal:.2f}  {pid}")
print("  -> F-controversial wins on a 300/290 split even though far bigger posts exist.\n")

# --- Best vs naive ratio: small samples must not beat large ones.
print("BEST (Wilson lower bound) vs naive upvote ratio:")
comments = [("one lucky vote", 1, 0), ("small sample", 9, 1),
            ("big and good", 400, 100), ("big and great", 900, 100)]
print(f"  {'comment':<16}{'ratio':>8}{'wilson':>9}")
for name, u, d in sorted(comments, key=lambda c: -confidence(c[1], c[2])):
    print(f"  {name:<16}{u/(u+d):>8.2f}{confidence(u, d):>9.3f}")
print("  -> naive ratio ranks '1 lucky vote' (1.00) first; Wilson correctly ranks it last,")
print("     because one sample tells us almost nothing about the true rate.")


## 2️⃣ Vote counter contention — the hot-key problem

When a post goes viral, millions of users hit the same row:

```
UPDATE posts SET ups = ups + 1 WHERE id = 42;   -- serialised on one row
```

Only one transaction at a time can hold the row lock. Every other vote queues up. Let's
measure it with threads + a single mutex as a stand-in for the row lock.


### 🐌 Bad — one global counter, one lock


In [ ]:
import threading, time

# A database row lock is not held for a few CPU cycles — it is held for the whole
# round trip to the storage engine (~1 ms). That is what makes it a bottleneck.
# We model the critical section with time.sleep(), which (like real I/O) releases
# the GIL, so this benchmark measures genuine lock contention rather than Python overhead.
ROW_LOCK_MS = 2.0 / 1000       # 2 ms of "database work" while holding the lock
N_THREADS   = 16
PER_THREAD  = 150              # 2,400 votes total — enough to see the difference

class SingleCounter:
    """One row, one lock. Every voter in the world queues behind it."""
    def __init__(self):
        self.n = 0
        self.lock = threading.Lock()
    def inc(self, user_id: int):
        with self.lock:                        # every thread fights for THIS lock
            time.sleep(ROW_LOCK_MS)            # the DB round trip, serialised
            self.n += 1

def run(counter, n_threads=N_THREADS, per_thread=PER_THREAD):
    t0 = time.time()
    def worker(tid):
        for i in range(per_thread):
            counter.inc(tid * 100_000 + i)
    threads = [threading.Thread(target=worker, args=(t,)) for t in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return time.time() - t0

single = SingleCounter()
dt_single = run(single)
print(f"single counter  : {single.n:>7,} votes in {dt_single:5.2f}s  -> {single.n/dt_single:>9,.0f} votes/s")
print(f"  (≈ 1 / {ROW_LOCK_MS*1000:.0f}ms = {1/ROW_LOCK_MS:,.0f}/s ceiling — adding threads cannot help)")


### ✅ Better — sharded counters (one lock per shard)


In [ ]:
class ShardedCounter:
    """N rows, N locks. `UPDATE vote_shards SET n=n+1 WHERE post_id=? AND shard=?`"""
    def __init__(self, n_shards: int = 64):
        self.shards = [0] * n_shards
        self.locks  = [threading.Lock() for _ in range(n_shards)]
    def inc(self, user_id: int):
        idx = user_id % len(self.shards)       # pick shard by user id -> even spread
        with self.locks[idx]:
            time.sleep(ROW_LOCK_MS)            # same simulated DB work
            self.shards[idx] += 1
    def total(self):
        return sum(self.shards)

sharded = ShardedCounter(n_shards=64)
dt_sharded = run(sharded)
print(f"64-shard counter: {sharded.total():>7,} votes in {dt_sharded:5.2f}s  -> {sharded.total()/dt_sharded:>9,.0f} votes/s")
print(f"speedup over single row: {dt_single/dt_sharded:.1f}x")
print("\nThe ceiling is now min(shards, concurrent writers) instead of 1. With 64 shards and")
print(f"{N_THREADS} writer threads, {N_THREADS} row locks are held at once instead of one.")


Sharding turns a single contended row into N less-contended rows, and the measured speedup
above is roughly `min(n_shards, concurrent_writers)`.

**What it costs — say this part out loud in an interview:**

- **Reads get more expensive.** Displaying `ups` now means `SELECT SUM(n) FROM vote_shards
  WHERE post_id=X` — 64 rows instead of 1. Acceptable only because reads are cached.
- **You cannot cheaply ask "how many shards does this post need?"** Provision 64 shards for
  every post and you've multiplied your row count by 64 for the 99.99% of posts that get
  eleven votes. Real systems shard *adaptively*: start at 1, promote a post to sharded
  counters once it crosses a write-rate threshold.
- **Atomicity across shards is gone.** There is no instant at which the sum is a consistent
  snapshot; you are reading a slightly-stale number by construction.

### 🏆 Best — write-behind to Redis, batch-flush to DB

In [ ]:
class WriteBehindCounter:
    """Increments land in memory; a flusher writes ONE batched row update per post.
    The DB round trip happens once per flush, not once per vote."""
    def __init__(self):
        self.buffer: dict[int, int] = {}
        self.lock = threading.Lock()
        self.durable: dict[int, int] = {}      # "the database"

    def inc(self, user_id: int, post_id: int = 42):
        with self.lock:                        # microseconds, no I/O inside
            self.buffer[post_id] = self.buffer.get(post_id, 0) + 1

    def flush(self):
        with self.lock:
            delta = self.buffer
            self.buffer = {}
        for pid, d in delta.items():
            time.sleep(ROW_LOCK_MS)            # ONE DB round trip for the whole batch
            self.durable[pid] = self.durable.get(pid, 0) + d

wb = WriteBehindCounter()
dt_wb = run(wb)                                # same 16 threads x 150 votes
wb.flush()
total = wb.durable[42]
print(f"write-behind    : {total:>7,} votes in {dt_wb:5.2f}s  -> {total/dt_wb:>9,.0f} votes/s")
print(f"speedup over single row: {dt_single/dt_wb:,.0f}x")
print(f"\nDB round trips: single={single.n:,}  sharded={sharded.total():,}  write-behind={len(wb.durable)}")
print("That last number is the point: {:,} votes collapsed into 1 write.".format(total))
print("\nThe cost: everything still in the buffer is lost if the process dies. For vote")
print("counts that is fine (they are an estimate anyway, and the votes table is the source")
print("of truth). For anything you must not lose — payments, inventory — it is not.")


| Approach | Throughput | Consistency | Complexity |
|---|---|---|---|
| Single counter | 💀 low | strong | trivial |
| Sharded counter (64) | 🔺 high | strong | small |
| Redis write-behind | 🔥 highest | eventual (seconds) | flusher + failover |

Reddit/Twitter/YouTube-scale systems all use the write-behind approach for like/view counts.


## 2️⃣b Vote fuzzing — why the number you see is a lie

Reddit deliberately **does not show you the true vote count**. Displayed scores are fuzzed, and
the up/down split shown on a post is partly fabricated. This is not a bug; it is anti-abuse.

The attack it defeats: a spammer with a botnet wants to know *which of their fake accounts got
banned*. They post something, vote with 100 sockpuppets, and read the counter. If it says +100,
all accounts still work. If it says +63, they know 37 are shadowbanned, and they can binary-search
to identify exactly which — then rotate them out and keep going.

Fuzzing breaks that feedback loop. If the displayed count is noisy, the attacker cannot tell
"my bot was banned" apart from "the counter is fuzzed today".

**The cost is real and worth stating out loud**, because an interviewer will ask:

- Vote totals shown to users no longer add up (`ups - downs ≠ score`), which confuses people
  and generates support tickets forever.
- Third-party analytics built on scraped scores are wrong.
- You must keep the **true** counts internally for ranking — fuzz only at the presentation
  layer, never in the ranking pipeline, or your sort becomes noise.
- The fuzz must be **deterministic per (post, viewer-bucket, time-bucket)**. Random noise on
  every request is trivially defeated: the attacker just polls 1,000 times and averages.

In [ ]:
import hashlib

TRUE_SCORE = 100          # what the ranking pipeline actually uses

def fuzz(post_id: str, true_score: int, time_bucket: int, spread: float = 0.10) -> int:
    """Deterministic display fuzz.

    Seeded on (post, time bucket, TRUE score). Two consequences that matter:
      * polling the same unchanged post always returns the same number, so an attacker
        cannot average the noise away;
      * a change in the true score RE-ROLLS the noise, so the observed delta is not the
        real delta. That is the property that breaks vote-confirmation attacks.
    """
    if true_score < 10:
        return true_score                     # don't fuzz tiny scores into nonsense
    seed = hashlib.sha256(f"{post_id}:{time_bucket}:{true_score}".encode()).digest()
    frac = int.from_bytes(seed[:2], "big") / 65535 * 2 - 1     # -1.0 .. +1.0
    return max(0, round(true_score * (1 + frac * spread)))

print("1) Determinism — the attacker polls the same post 6 times in one time bucket:")
print(f"   {[fuzz('post-A', TRUE_SCORE, 42) for _ in range(6)]}")
print("   Identical every time, so polling-and-averaging recovers nothing.\n")

print("2) The signal fuzzing actually destroys: 'did my last few votes count?'")
print("   The attacker adds sockpuppet votes one at a time and watches the counter.")
print(f"   {'true':>6}{'displayed':>11}{'observed delta':>16}")
prev = None
for true in range(100, 108):
    shown = fuzz("post-A", true, 42)
    delta = "-" if prev is None else f"{shown - prev:+d}"
    print(f"   {true:>6}{shown:>11}{delta:>16}")
    prev = shown
print("   The observed deltas are not +1. The attacker cannot confirm that any specific")
print("   sockpuppet's vote landed, so they cannot binary-search for their banned accounts.\n")

print("3) Fuzzing is a DISPLAY concern only:")
print(f"   displayed to users : {fuzz('post-A', TRUE_SCORE, 42)}")
print(f"   fed to hot ranking : {TRUE_SCORE}   <- never fuzzed, or the sort becomes noise")
print("\n   Note what fuzzing does NOT do: a ±10% spread cannot hide a large change. If half")
print("   the botnet is banned, the displayed score still visibly halves. Fuzzing defeats")
print("   *precise* inference, not *coarse* inference — a limitation worth stating honestly.")


## 3️⃣ Serving comment trees

Our `comments` table has both `parent_id` (adjacency list) and `path` (materialized path)
from Notebook 2. Let's see the three ways to load a subtree.


In [ ]:
# Fresh SQLite DB with a bigger fake tree so timing is meaningful.
import sqlite3, random, time

db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.executescript('''
    CREATE TABLE comments (
        id INTEGER PRIMARY KEY,
        post_id INTEGER,
        parent_id INTEGER,
        body TEXT,
        path TEXT
    );
    CREATE INDEX ix_comments_post_parent ON comments(post_id, parent_id);
    CREATE INDEX ix_comments_path        ON comments(path);

    -- Closure table: (ancestor, descendant, depth)
    CREATE TABLE comment_ancestry (
        ancestor   INTEGER,
        descendant INTEGER,
        depth      INTEGER,
        PRIMARY KEY (ancestor, descendant)
    );
''')

random.seed(1)

def insert_comment(post_id, parent_id, body):
    cur = db.execute("INSERT INTO comments(post_id, parent_id, body) VALUES (?,?,?)",
                     (post_id, parent_id, body))
    cid = cur.lastrowid
    if parent_id is None:
        path = f"/{cid}/"
    else:
        parent_path = db.execute("SELECT path FROM comments WHERE id=?", (parent_id,)).fetchone()["path"]
        path = f"{parent_path}{cid}/"
    db.execute("UPDATE comments SET path=? WHERE id=?", (path, cid))

    # closure-table rows: self + all ancestor rows
    db.execute("INSERT INTO comment_ancestry VALUES (?,?,0)", (cid, cid))
    if parent_id is not None:
        db.execute('''INSERT INTO comment_ancestry(ancestor, descendant, depth)
                      SELECT ancestor, ?, depth+1 FROM comment_ancestry WHERE descendant=?''',
                   (cid, parent_id))
    return cid

# Grow a random tree of 5 000 comments under post #1
roots = [insert_comment(1, None, f"root {i}") for i in range(20)]
all_ids = list(roots)
for i in range(5_000):
    parent = random.choice(all_ids)
    all_ids.append(insert_comment(1, parent, f"reply {i}"))
db.commit()
print("comments inserted:", db.execute("SELECT COUNT(*) FROM comments").fetchone()[0])


### 🐌 Bad — walk `parent_id` in Python (N+1 queries)


In [ ]:
def fetch_subtree_py(root_id: int):
    out = []
    stack = [root_id]
    while stack:
        cid = stack.pop()
        row = db.execute("SELECT id, body FROM comments WHERE id=?", (cid,)).fetchone()
        out.append(row["body"])
        kids = db.execute("SELECT id FROM comments WHERE parent_id=?", (cid,)).fetchall()
        stack.extend(k["id"] for k in kids)
    return out

root = roots[0]
t0 = time.time(); rows = fetch_subtree_py(root); dt = (time.time()-t0)*1000
print(f"N+1 walk         : {len(rows):>5} rows in {dt:6.1f} ms")


### ✅ Better — one indexed prefix query via materialized path


In [ ]:
def fetch_subtree_path(root_id: int):
    path = db.execute("SELECT path FROM comments WHERE id=?", (root_id,)).fetchone()["path"]
    return db.execute("SELECT body FROM comments WHERE path LIKE ? ORDER BY path",
                      (path + "%",)).fetchall()

t0 = time.time(); rows = fetch_subtree_path(root); dt = (time.time()-t0)*1000
print(f"materialized path: {len(rows):>5} rows in {dt:6.1f} ms")


### 🏆 Best — closure table join (flexible: filter by depth too)


In [ ]:
def fetch_subtree_closure(root_id: int, max_depth: int | None = None):
    q = '''SELECT c.body
           FROM comment_ancestry a
           JOIN comments c ON c.id = a.descendant
           WHERE a.ancestor = ?'''
    args = [root_id]
    if max_depth is not None:
        q += " AND a.depth <= ?"; args.append(max_depth)
    return db.execute(q, args).fetchall()

t0 = time.time(); rows = fetch_subtree_closure(root); dt = (time.time()-t0)*1000
print(f"closure table    : {len(rows):>5} rows in {dt:6.1f} ms")

# Closure wins when you want "only the first 3 levels of replies" —
# a materialized path can't do that without string-counting '/'.
t0 = time.time(); rows = fetch_subtree_closure(root, max_depth=2); dt = (time.time()-t0)*1000
print(f"closure (depth≤2): {len(rows):>5} rows in {dt:6.1f} ms")


### Pick-your-tree cheat sheet

| Pattern | Read subtree | Read level-limited | Move/rewrite a branch | Storage |
|---|---|---|---|---|
| `parent_id` only | ❌ recursion | ❌ recursion | ✅ cheap | 🟢 tiny |
| Materialized path | ✅ fast | ⚠️ string math | ❌ rewrite all descendants | 🟡 |
| Closure table | ✅ fast | ✅ trivial | ⚠️ rewrite ancestry rows | 🔴 largest |

In practice Reddit **caches** the rendered comment JSON per `(post_id, sort)` in Redis and
invalidates on new comment / vote. The DB layout above is for the *cache-miss* path.


## 📚 Summary

1. **Hot ranking = log(votes) + age_bonus.** The score of an existing post never changes for
   fixed votes, so we only re-score *new* votes. No expensive global decay pass.
2. **Don't fight a single row** under vote storms. Shard counters — or, even better, buffer
   in Redis and flush to the DB in batches. Eventual consistency is fine for vote counts.
3. **Don't walk parent pointers** on hot paths. Use a materialized path (fast subtree) or a
   closure table (flexible depth filters) — both turn a recursive problem into one index scan.

### 💡 Interview tips

- If asked "how do you rank Hot?", lead with: *"log(|ups − downs|) + sign × age / 45000 — the
  key property is that existing posts' scores don't change, so we only re-rank on new votes."*
- For vote contention, mention **sharded counters** *and* **write-behind** — showing both says
  you understand the levels of scale.
- For comments, mention **cached pre-rendered JSON per (post, sort)** — Reddit's secret sauce.

### Where to go next

- `04-patterns/` — Cache-Aside, Write-Behind, and Materialized View are the patterns powering this lab.
- `06-system-designs/fb-news-feed/` — different ranking problem (per-user relevance vs global popularity).
- `06-system-designs/top-k/` — generalises "Hot" to any streaming top-K problem.
